# 4. Installing security cameras: minimum vertex cover

Lecture: *Week 6 - Integer Linear Programming - Further Problems*.

*Original script: `5 Source Files/Python/security_cameras_vertex_cover.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same optimal value (4 cameras).*

*Note: there is more than one minimum cover, so the solver may pick a different set of nodes from other runs or solvers (GLPK, for example, picks 0, 2, 4, 6). Any cover that uses 4 cameras is optimal.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

In [ ]:
from pyomo.environ import *

In [ ]:
# Define the binary variables
model = ConcreteModel()
A = [
    [0, 1, 0, 0, 0, 0, 0, 1],
    [0, 0, 1, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 1],
    [0, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 0]
]

no_vars = len(A[0])	# variables are the number of columns in a row

model.x = Var(range(no_vars), within=Binary)

In [ ]:
# Create the optimization problem
model.obj = Objective(expr=sum(model.x[i] for i in range(no_vars)), sense=minimize)

In [ ]:
# Define the inequality constraints
model.vertex = ConstraintList()
for i in range(no_vars):
    for j in range(no_vars):
        if A[i][j]:
            model.vertex.add(model.x[i] + model.x[j] >= 1)

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')

In [ ]:
# Display the model
model.pprint()

In [ ]:
# Solve the model
solver.solve(model, tee=False)  # Set tee=True to display solver output

In [ ]:
# Display the results
model.display()